In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from typing import Dict, List, Tuple, Optional
import logging
from pathlib import Path
from tqdm import tqdm
import numpy as np
import os
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import os
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split

%matplotlib inline

In [ ]:
class LungCancerDataset(Dataset):
    
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.classes = ['Bengin cases', 'Malignant cases', 'Normal cases']
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}

        self.images = []
        self.labels = []

        # Load all images and labels
        for class_name in self.classes:
            class_dir = os.path.join(data_dir, class_name)
            if os.path.exists(class_dir):
                for img_file in os.listdir(class_dir):
                    if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.images.append(os.path.join(class_dir, img_file))
                        self.labels.append(self.class_to_idx[class_name])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
import torch

# ---------- CLAHE Transform ----------
class CLAHETransform:
    """Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)"""
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, image):
        if isinstance(image, Image.Image):
            image = np.array(image)

        # Convert to LAB and apply CLAHE to L channel
        if len(image.shape) == 3:  
            lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
            clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
            lab[:,:,0] = clahe.apply(lab[:,:,0])
            image = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        else:  
            clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
            image = clahe.apply(image)

        return Image.fromarray(image)

# ---------- Data Transforms ----------
def get_data_transforms():
    train_transforms = transforms.Compose([
        CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),   # CLAHE
        transforms.Resize((224, 224)),                           # Resize
        transforms.RandomRotation(15),                           # Rotation
        transforms.RandomAffine(degrees=0,                       # Shift, zoom, shear
                                translate=(0.1, 0.1), 
                                scale=(0.9, 1.1), 
                                shear=10, 
                                fill=0),                        # fill pixels with 0 (black)
        transforms.RandomHorizontalFlip(p=0.5),                  # Flip
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                             std=[0.229, 0.224, 0.225])
    ])

    test_transforms = transforms.Compose([
        CLAHETransform(clip_limit=2.0, tile_grid_size=(8, 8)),   # CLAHE
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                             std=[0.229, 0.224, 0.225])
    ])

    return train_transforms, test_transforms

# ---------- Usage ----------
train_transforms, test_transforms = get_data_transforms()

print("Loading dataset...")
full_dataset = LungCancerDataset('/kaggle/input/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset', transform=train_transforms)

# Train-test split (80-20)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

# Assign test transform
test_dataset.dataset.transform = test_transforms

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

# print("Creating CCAL model...")
# model = MainModel(
#     num_classes=3,
#     num_heads=config['num_heads'],
#     key_dim=config['key_dim'],
#     val_dim=config['value_dim']
# )


In [ ]:
for idx, (img, lbl) in enumerate(full_dataset):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    
    # unnormalize: x * std + mean
    img = img.clone()  # avoid modifying original
    for c in range(3):
        img[c] = img[c] * std[c] + mean[c]
    
    # convert to numpy for plotting
    img = img.permute(1, 2, 0).numpy()  # (H, W, C)
    img = img.clip(0, 1)  # keep values in [0,1]
    
    plt.imshow(img)
    plt.axis("off")
    plt.show()


In [ ]:
class CNN_Model1(nn.Module):
    def __init__(self,inp_chn = 3, out_chn=32):
        super(CNN_Model1,self).__init__()
        self.conv1 = nn.Conv2d(inp_chn,out_chn,kernel_size=3,padding=1)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(out_chn,out_chn*2,kernel_size=3,padding=1)
        self.conv3 = nn.Conv2d(out_chn*2,out_chn*2,kernel_size=3,padding=1)

        self.conv4 = nn.Conv2d(out_chn*2,out_chn*4,kernel_size=3,padding=1)
        self.conv5= nn.Conv2d(out_chn*4,out_chn*4,kernel_size=3,padding=1)
        
        self.conv6 = nn.Conv2d(out_chn*4,out_chn*8,kernel_size=3,padding=1)
        self.global_max_pool = nn.AdaptiveMaxPool2d(1)

    def forward(self,x):
        # print(x.shape)
        x = self.relu(self.conv1(x))
        # print(x.shape)
        x = self.pool(x)
        # print(x.shape)
        x = self.relu(self.conv2(x))
        # print(x.shape)
        x = self.relu(self.conv3(x))
        # print(x.shape)
        x = self.pool(x)
        # print(x.shape)
        x = self.relu(self.conv4(x))
        # print(x.shape)
        x = self.relu(self.conv5(x))
        # print(x.shape)
        x = self.pool(x)
        # print(x.shape)
        x = self.relu(self.conv6(x))
        # print(x.shape)
        x = self.global_max_pool(x)
        # print(x.shape)

        print(x.view(x.shape[0],-1).shape)
        return x.view(x.shape[0],-1)

class CNN_Model2(nn.Module):
    def __init__(self,inp_chn = 3, out_chn=16):
        super(CNN_Model2,self).__init__()
        self.conv1 = nn.Conv2d(inp_chn,out_chn,kernel_size=3,padding=1)

        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(out_chn,out_chn*2,kernel_size=3,padding=1)
        self.global_max_pool = nn.AdaptiveMaxPool2d(1)

    def forward(self,x):
        # print(x.shape)
        x = self.relu(self.conv1(x))
        # print(x.shape)
        x = self.pool(x)
        # print(x.shape)
        x = self.relu(self.conv2(x))
        # print(x.shape)
        x = self.pool(x)
        # print(x.shape)
        x = self.global_max_pool(x)
        # print(x.shape)
        print(x.view(x.shape[0],-1).shape)
        return x.view(x.shape[0],-1)


class MainModel(nn.Module):
    def __init__(self, num_classes = 3, num_heads=4, key_dim = 16, val_dim = 64):
        super(MainModel, self).__init__()

        self.cnn1= CNN_Model1()
        self.cnn2 = CNN_Model2()

        self.concat_dim = 256+32

        self.embed_dim = num_heads*key_dim
        self.feature_projection = nn.Linear(self.concat_dim,self.embed_dim)
        self.multihead_attn = nn.MultiheadAttention(
            embed_dim = self.embed_dim,
            num_heads = num_heads,
            dropout=0.1,
            batch_first=True
        )
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.batch_norm1 = nn.BatchNorm1d(self.embed_dim)

        self.dense1 = nn.Linear(self.embed_dim,256)
        self.gelu = nn.GELU()
        self.batch_norm2 = nn.BatchNorm1d(256)
        self.dropout= nn.Dropout(0.2)

        self.ffn = nn.Linear(256, num_classes)
        

    def forward(self, x): 

        cnn1_features = self.cnn1(x)
        cnn2_features = self.cnn2(x)

        concat_features = torch.cat([cnn1_features , cnn2_features],dim=1)

        projected_features = self.feature_projection(concat_features)

        #added sequence for attention
        attn_in = projected_features.unsqueeze(1)

        attn_out , attn_weights = self.multihead_attn(
            attn_in , attn_in, attn_in
        )

        attn_out = attn_out.transpose(1, 2)  # (batch_size, 64, 1)
        
        # Apply Global Average Pooling 1D
        x = self.global_avg_pool(attn_out)  # (batch_size, 64, 1)
        

        x = x.squeeze(-1)  # (batch_size, 64)
        
        x = self.batch_norm1(x)
        x =  self.dense1(x)
        x = self.gelu(x)
        x = self.batch_norm2(x)
        x = self.dropout(x)
        output = self.ffn(x)
        return output, attn_weights
        
        

In [ ]:
train_transforms, test_transforms = get_data_transforms()
full_dataset = LungCancerDataset('/kaggle/input/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset', transform=train_transforms)


In [ ]:
len(full_dataset)

In [ ]:
for idx, (img, lbl) in enumerate(full_dataset):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    
    # unnormalize: x * std + mean
    img = img.clone()  # avoid modifying original
    for c in range(3):
        img[c] = img[c] * std[c] + mean[c]
    
    # convert to numpy for plotting
    img = img.permute(1, 2, 0).numpy()  # (H, W, C)
    img = img.clip(0, 1)  # keep values in [0,1]
    
    plt.imshow(img)
    plt.axis("off")
    plt.show()


In [ ]:
dataset_path = "/kaggle/input/the-iqothnccd-lung-cancer-dataset/The IQ-OTHNCCD lung cancer dataset"
data = {}
for path in os.listdir(dataset_path):
    if path[-3:]!='txt':
        if path == "Bengin cases":
            data["Benign cases"] = []
            class_path = os.path.join(dataset_path, path)
            for img_path in os.listdir(class_path):
                full_path = os.path.join(class_path, img_path)
                data["Benign cases"].append(full_path) 
        else:
            data[path] = []
            class_path = os.path.join(dataset_path, path)
            for img_path in os.listdir(class_path):
                full_path = os.path.join(class_path, img_path)
                data[path].append(full_path) 
                # break 
    else :
        continue

In [ ]:
def show_one_img(path,cls):
    img = Image.open(path)
    plt.imshow(img,cmap='gray')
    plt.axis('off')  
    plt.title(cls) 
    plt.show()
    print(img.size)

        

In [ ]:
for cls,paths in data.items():
    show_one_img(paths[0],cls)

In [ ]:
import random

def show_random_images(data, class_name, n_img=5, rows=1, cols=5, figsize=(10, 5)):
    paths = data[class_name]
    if not paths:
        print(f"No images found for class: {class_name}")
        return

    selected = random.sample(paths, min(n_img, len(paths))) 

    plt.figure(figsize=figsize)
    for i, path in enumerate(selected):
        plt.subplot(rows, cols, i + 1)
        img = Image.open(path)
        plt.imshow(img, cmap='gray')
        plt.axis('off')
        plt.title(class_name)

    plt.tight_layout()
    plt.show()


In [ ]:
show_random_images(data, "Malignant cases", n_img=5)

In [ ]:
class LungCancerDataset(Dataset):
    def __init__(self, data_dict, transform=None):
        self.samples = []
        self.transform = transform
        self.class_to_idx = {
            "Normal cases": 0, 
            "Benign cases": 1, 
            "Malignant cases": 2
        }
        
        # Build samples list
        for cls, paths in data_dict.items():
            for path in paths:
                self.samples.append((path, self.class_to_idx[cls]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        image = np.array(image)
        
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]
        
        return image, label

In [ ]:
dataset = LungCancerDataset(data)

In [ ]:
for img, cls in dataset:
    plt.imshow(img)
    print( cls)
    break

In [ ]:
# Albumentations transforms - customize these as needed
train_transform = A.Compose([
    A.Resize(224, 224),                    # Match your dataset resolution
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.4),
    A.RandomBrightnessContrast(
        brightness_limit=0.1, 
        contrast_limit=0.1, 
        p=0.3
    ),
    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.1,
        rotate_limit=15,
        p=0.3
    ),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

test_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# Create datasets and dataloaders
def create_dataloaders(data_dict, batch_size=16, test_size=0.2, num_workers=4):
    # Create full dataset
    full_dataset = LungCancerDataset(data_dict, transform=None)
    
    # Split indices for train/test
    indices = list(range(len(full_dataset)))
    labels = [full_dataset.samples[i][1] for i in indices]
    
    train_idx, test_idx = train_test_split(
        indices, 
        test_size=test_size, 
        stratify=labels,  # Maintain class distribution
        random_state=42
    )
    
    # Create separate datasets with appropriate transforms
    train_samples = [full_dataset.samples[i] for i in train_idx]
    test_samples = [full_dataset.samples[i] for i in test_idx]
    
    train_data_dict = {"Normal cases": [] ,"Benign cases": [], "Malignant cases": []}
    test_data_dict = {"Normal cases" :[],"Benign cases": [], "Malignant cases": []}
    
    idx_to_class = {0:"Normal cases", 1:  "Benign cases" , 2: "Malignant cases"}
    
    for path, label in train_samples:
        train_data_dict[idx_to_class[label]].append(path)
    
    for path, label in test_samples:
        test_data_dict[idx_to_class[label]].append(path)
    
    # Create datasets with transforms
    train_dataset = LungCancerDataset(train_data_dict, transform=train_transform)
    test_dataset = LungCancerDataset(test_data_dict, transform=test_transform)
    
    # Create dataloaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )
    
    return train_loader, test_loader, train_dataset, test_dataset

# Create dataloaders
train_loader, test_loader, train_dataset, test_dataset = create_dataloaders(
    data, 
    batch_size=32, 
    test_size=0.2
)

# Print dataset info
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Number of classes: {len(set(train_dataset.class_to_idx.values()))}")

# Test a batch
for images, labels in train_loader:
    print(f"Batch shape: {images.shape}")
    print(f"Labels shape: {labels.shape}")
    break


In [ ]:
# Data preprocessing according to the paper
def get_data_transforms():
    """Data transforms including CLAHE-like enhancement and augmentation"""
    train_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        # Data augmentation as mentioned in the paper
        transforms.RandomRotation(10),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    test_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return train_transforms, test_transforms


In [ ]:
# Create the CCAL (Concatenated CNN Attention Lung) model implementation in PyTorch
# Based on the paper analysis, this model combines two custom CNNs with multi-head attention

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
import torchvision.models as models

class CustomCNN1(nn.Module):
    """First CNN model with 4 convolutional layers for deeper feature extraction"""
    def __init__(self):
        super(CustomCNN1, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)
        
        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.global_pool = nn.AdaptiveMaxPool2d(1)
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool3(x)
        
        x = F.relu(self.conv5(x))
        x = self.global_pool(x)
        return x.view(x.size(0), -1)


class CustomCNN2(nn.Module):
    """Second CNN model with 2 convolutional layers for broader feature extraction"""
    def __init__(self):
        super(CustomCNN2, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        self.global_pool = nn.AdaptiveMaxPool2d(1)
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        
        x = self.global_pool(x)
        return x.view(x.size(0), -1)


class CCALModel(nn.Module):
    """Concatenated CNN Attention Lung cancer classification model"""
    def __init__(self, num_classes=3, num_heads=4, key_dim=16, value_dim=64):
        super(CCALModel, self).__init__()
        
        # Two custom CNN models
        self.cnn1 = CustomCNN1()
        self.cnn2 = CustomCNN2()
        
        # Concatenated feature dimension
        concat_dim = 256 + 32  # CNN1 output + CNN2 output = 288
        
        # Reshape for attention mechanism
        self.feature_reshape = nn.Linear(concat_dim, key_dim * value_dim)
        
        # Multi-head attention
        self.multihead_attn = nn.MultiheadAttention(
            embed_dim=key_dim * value_dim,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True
        )
        
        # Global Average Pooling 1D (equivalent to the paper's approach)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        
        # Batch normalization
        self.batch_norm1 = nn.BatchNorm1d(key_dim * value_dim)
        
        # MLP block
        self.dense1 = nn.Linear(key_dim * value_dim, 256)
        self.batch_norm2 = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(0.2)
        
        # Final classification layer
        self.classifier = nn.Linear(256, num_classes)
        
    def forward(self, x):
        # Extract features from both CNN models
        cnn1_features = self.cnn1(x)  # Shape: (batch_size, 256)
        cnn2_features = self.cnn2(x)  # Shape: (batch_size, 32)
        
        # Concatenate features
        concat_features = torch.cat([cnn1_features, cnn2_features], dim=1)  # Shape: (batch_size, 288)
        
        # Reshape for attention mechanism
        reshaped_features = self.feature_reshape(concat_features)  # Shape: (batch_size, key_dim * value_dim)
        
        # Add sequence dimension for attention (treating each feature as a sequence element)
        seq_features = reshaped_features.unsqueeze(1)  # Shape: (batch_size, 1, key_dim * value_dim)
        
        # Multi-head attention (self-attention)
        attn_output, attn_weights = self.multihead_attn(seq_features, seq_features, seq_features)
        
        # Remove sequence dimension and apply global average pooling
        attn_output = attn_output.squeeze(1)  # Shape: (batch_size, key_dim * value_dim)
        
        # Batch normalization
        attn_output = self.batch_norm1(attn_output)
        
        # MLP block with GELU activation (similar to paper)
        x = F.gelu(self.dense1(attn_output))
        x = self.batch_norm2(x)
        x = self.dropout(x)
        
        # Final classification
        output = F.softmax(self.classifier(x), dim=1)
        
        return output, attn_weights


# Data preprocessing according to the paper
def get_data_transforms():
    """Data transforms including CLAHE-like enhancement and augmentation"""
    train_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        # Data augmentation as mentioned in the paper
        transforms.RandomRotation(10),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    test_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    return train_transforms, test_transforms


# Training configuration as per the paper
def get_training_config():
    """Training hyperparameters from the paper"""
    config = {
        'batch_size': 32,
        'learning_rate': 0.001,
        'epochs': 50,
        'optimizer': 'adam',
        'loss_function': 'sparse_categorical_crossentropy',
        'train_split': 0.8,
        'test_split': 0.2,
        'num_heads': 4,
        'key_dim': 16,
        'value_dim': 64
    }
    return config


# Example usage and model creation
def create_model():
    """Create the CCAL model with paper specifications"""
    config = get_training_config()
    model = CCALModel(
        num_classes=3,
        num_heads=config['num_heads'],
        key_dim=config['key_dim'],
        value_dim=config['value_dim']
    )
    return model, config


# Print model architecture
model, config = create_model()
print("CCAL Model Architecture:")
print(model)
print(f"\nModel Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:


import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
import cv2
import numpy as np
import os
import pandas as pd
from PIL import Image
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')


class CLAHETransform:
    """Apply CLAHE (Contrast Limited Adaptive Histogram Equalization) as mentioned in the paper"""
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, image):
        if isinstance(image, Image.Image):
            image = np.array(image)

        if len(image.shape) == 3:
            # Convert RGB to LAB, apply CLAHE to L channel, convert back
            lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
            clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
            lab[:,:,0] = clahe.apply(lab[:,:,0])
            image = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        else:
            clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
            image = clahe.apply(image)

        return Image.fromarray(image)


class LungCancerDataset(Dataset):
    """Custom dataset for IQ-OTH/NCCD lung cancer dataset"""
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.classes = ['Bengin cases', 'Malignant cases', 'Normal cases']
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}

        self.images = []
        self.labels = []

        # Load all images and labels
        for class_name in self.classes:
            class_dir = os.path.join(data_dir, class_name)
            if os.path.exists(class_dir):
                for img_file in os.listdir(class_dir):
                    if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.images.append(os.path.join(class_dir, img_file))
                        self.labels.append(self.class_to_idx[class_name])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label


class CustomCNN1(nn.Module):
    """First CNN model with 4 convolutional layers for hierarchical feature extraction"""
    def __init__(self):
        super(CustomCNN1, self).__init__()

        # First block
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Second block
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Third block (two conv layers)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv4 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)

        # Fourth block
        self.conv5 = nn.Conv2d(128, 256, kernel_size=3, padding=1)

        # Global max pooling as mentioned in paper
        self.global_pool = nn.AdaptiveMaxPool2d(1)

    def forward(self, x):
        # First block
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        # Second block
        x = F.relu(self.conv2(x))
        x = self.pool2(x)

        # Third block
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = self.pool3(x)

        # Fourth block
        x = F.relu(self.conv5(x))

        # Global pooling
        x = self.global_pool(x)
        return x.view(x.size(0), -1)


class CustomCNN2(nn.Module):
    """Second CNN model with 2 convolutional layers for broader pattern recognition"""
    def __init__(self):
        super(CustomCNN2, self).__init__()

        # First block
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Second block
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Global max pooling
        self.global_pool = nn.AdaptiveMaxPool2d(1)

    def forward(self, x):
        # First block
        x = F.relu(self.conv1(x))
        x = self.pool1(x)

        # Second block
        x = F.relu(self.conv2(x))
        x = self.pool2(x)

        # Global pooling
        x = self.global_pool(x)
        return x.view(x.size(0), -1)


class CCALModel(nn.Module):
    """
    Concatenated CNN Attention Lung cancer classification model
    Architecture: CNN1 + CNN2 -> Concatenation -> Multi-Head Attention -> MLP -> Classification
    """
    def __init__(self, num_classes=3, num_heads=4, key_dim=16, value_dim=64):
        super(CCALModel, self).__init__()

        # Two custom CNN models
        self.cnn1 = CustomCNN1()  # Output: 256 features
        self.cnn2 = CustomCNN2()  # Output: 32 features

        # Concatenated feature dimension
        self.concat_dim = 256 + 32  # 288 total features

        # Prepare for multi-head attention
        self.attention_dim = num_heads * key_dim  # Should be divisible by num_heads
        self.feature_projection = nn.Linear(self.concat_dim, self.attention_dim)

        # Multi-head attention layer
        self.multihead_attention = nn.MultiheadAttention(
            embed_dim=self.attention_dim,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True
        )

        # Global Average Pooling (as mentioned in paper)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)

        # Batch normalization
        self.batch_norm1 = nn.BatchNorm1d(self.attention_dim)

        # MLP block with GELU activation (as mentioned in paper)
        self.dense1 = nn.Linear(self.attention_dim, 256)
        self.gelu = nn.GELU()
        self.batch_norm2 = nn.BatchNorm1d(256)
        self.dropout = nn.Dropout(0.2)

        # Final classification layer
        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        batch_size = x.size(0)

        # Extract features from both CNN models
        cnn1_features = self.cnn1(x)  # Shape: (batch_size, 256)
        cnn2_features = self.cnn2(x)  # Shape: (batch_size, 32)

        # Concatenate features
        concat_features = torch.cat([cnn1_features, cnn2_features], dim=1)

        # Project to attention dimension
        projected_features = self.feature_projection(concat_features)

        # Reshape for attention mechanism (add sequence dimension)
        attention_input = projected_features.unsqueeze(1)  # Shape: (batch_size, 1, attention_dim)

        # Apply multi-head attention (self-attention)
        attention_output, attention_weights = self.multihead_attention(
            attention_input, attention_input, attention_input
        )

        # Remove sequence dimension
        attention_output = attention_output.squeeze(1)  # Shape: (batch_size, attention_dim)

        # Batch normalization
        x = self.batch_norm1(attention_output)

        # MLP block
        x = self.dense1(x)
        x = self.gelu(x)
        x = self.batch_norm2(x)
        x = self.dropout(x)

        # Final classification
        output = self.classifier(x)

        return output, attention_weights


def get_data_transforms():
    """Data transforms including CLAHE and augmentation as mentioned in the paper"""

    train_transforms = transforms.Compose([
        CLAHETransform(clip_limit=2.0),  # CLAHE preprocessing
        transforms.Resize((224, 224)),
        # Data augmentation as mentioned in the paper
        transforms.RandomRotation(15),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=10),
        transforms.RandomHorizontalFlip(0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    test_transforms = transforms.Compose([
        CLAHETransform(clip_limit=2.0),  # CLAHE preprocessing
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    return train_transforms, test_transforms


def train_model(model, train_loader, val_loader, num_epochs=50, learning_rate=0.001, device='cuda'):
    """Training function following the paper's methodology"""

    # Loss function and optimizer as mentioned in paper
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Learning rate scheduler
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

    model.to(device)

    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
        for batch_idx, (data, targets) in enumerate(train_pbar):
            data, targets = data.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs, attention_weights = model(data)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total_train += targets.size(0)
            correct_train += predicted.eq(targets).sum().item()

            train_pbar.set_postfix({
                'Loss': f'{running_loss/(batch_idx+1):.4f}',
                'Acc': f'{100.*correct_train/total_train:.2f}%'
            })

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for data, targets in val_loader:
                data, targets = data.to(device), targets.to(device)
                outputs, _ = model(data)
                loss = criterion(outputs, targets)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total_val += targets.size(0)
                correct_val += predicted.eq(targets).sum().item()

        # Calculate metrics
        train_loss = running_loss / len(train_loader)
        train_acc = 100. * correct_train / total_train
        val_loss = val_loss / len(val_loader)
        val_acc = 100. * correct_val / total_val

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(val_acc)

        print(f'Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')

        scheduler.step()

    return train_losses, val_losses, train_accuracies, val_accuracies


def evaluate_model(model, test_loader, device='cuda'):
    """Comprehensive model evaluation with metrics from the paper"""
    model.eval()
    all_predictions = []
    all_targets = []
    all_probabilities = []

    with torch.no_grad():
        for data, targets in tqdm(test_loader, desc='Evaluating'):
            data, targets = data.to(device), targets.to(device)
            outputs, _ = model(data)
            probabilities = F.softmax(outputs, dim=1)

            _, predicted = outputs.max(1)

            all_predictions.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())

    # Calculate metrics as mentioned in the paper
    accuracy = accuracy_score(all_targets, all_predictions)
    precision = precision_score(all_targets, all_predictions, average='weighted')
    recall = recall_score(all_targets, all_predictions, average='weighted')
    f1 = f1_score(all_targets, all_predictions, average='weighted')

    # ROC-AUC for multiclass
    all_probabilities = np.array(all_probabilities)
    auc = roc_auc_score(all_targets, all_probabilities, multi_class='ovr', average='weighted')

    print(f"Test Results:")
    print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
    print(f"Recall: {recall:.4f} ({recall*100:.2f}%)")
    print(f"F1-Score: {f1:.4f} ({f1*100:.2f}%)")
    print(f"AUC-ROC: {auc:.4f} ({auc*100:.2f}%)")

    # Confusion Matrix
    cm = confusion_matrix(all_targets, all_predictions)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

    return accuracy, precision, recall, f1, auc


def plot_training_history(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training history as shown in the paper"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Loss plot
    ax1.plot(train_losses, label='Train Loss', color='blue')
    ax1.plot(val_losses, label='Validation Loss', color='red')
    ax1.set_title('Model Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)

    # Accuracy plot
    ax2.plot(train_accuracies, label='Train Accuracy', color='blue')
    ax2.plot(val_accuracies, label='Validation Accuracy', color='red')
    ax2.set_title('Model Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()


def main():
    """Main training and evaluation pipeline"""

    # Configuration (matching the paper)
    config = {
        'data_dir': '/path/to/IQ-OTH_NCCD_dataset',  # Update with your dataset path
        'batch_size': 32,
        'learning_rate': 0.001,
        'epochs': 50,
        'num_heads': 4,
        'key_dim': 16,
        'value_dim': 64,
        'device': 'cuda' if torch.cuda.is_available() else 'cpu'
    }

    print(f"Using device: {config['device']}")

    # Get data transforms
    train_transforms, test_transforms = get_data_transforms()

    # Load dataset
    print("Loading dataset...")
    full_dataset = LungCancerDataset(config['data_dir'], transform=train_transforms)

    # Split dataset (80% train, 20% test as mentioned in paper)
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

    # Update test dataset transform
    test_dataset.dataset.transform = test_transforms

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=4)

    # Create model
    print("Creating CCAL model...")
    model = CCALModel(
        num_classes=3,
        num_heads=config['num_heads'],
        key_dim=config['key_dim'],
        value_dim=config['value_dim']
    )

    print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

    # Split train into train and validation
    val_size = int(0.2 * len(train_dataset))
    train_size = len(train_dataset) - val_size
    train_subset, val_subset = random_split(train_dataset, [train_size, val_size])

    train_loader = DataLoader(train_subset, batch_size=config['batch_size'], shuffle=True, num_workers=4)
    val_loader = DataLoader(val_subset, batch_size=config['batch_size'], shuffle=False, num_workers=4)

    # Train model
    print("Starting training...")
    train_losses, val_losses, train_accuracies, val_accuracies = train_model(
        model, train_loader, val_loader, 
        num_epochs=config['epochs'], 
        learning_rate=config['learning_rate'],
        device=config['device']
    )

    # Plot training history
    plot_training_history(train_losses, val_losses, train_accuracies, val_accuracies)

    # Evaluate on test set
    print("Evaluating model...")
    accuracy, precision, recall, f1, auc = evaluate_model(model, test_loader, device=config['device'])

    # Save model
    torch.save(model.state_dict(), 'ccal_model.pth')
    print("Model saved as 'ccal_model.pth'")


if __name__ == "__main__":
    main()
